# no-relu-on-final-layer — ex1: diagnose and strip a stray ReLU on the classifier head

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `no-relu-on-final-layer`. Running the final beacon cell reports progress against the `CNN: No-ReLU on final layer` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `CNN: No-ReLU on final layer` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`no-relu-on-final-layer`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "no-relu-on-final-layer"
DD_SUBTOPIC = "CNN: No-ReLU on final layer"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## No ReLU on the final classifier layer — quick refresher

A classification CNN ends with a linear layer that produces **logits**: real numbers per class. The next step is `F.cross_entropy`, which internally applies `LogSoftmax` to the logits before computing the loss.

**Why no ReLU on the final layer.** ReLU clips negatives to 0. Once applied to logits:

- All previously negative logits become 0.
- `softmax([0, 0, ..., 0, big_positive]) → [near_uniform, ..., dominated_by_big]`.
- Worse: if **every** logit is negative pre-ReLU, ReLU produces an all-zero vector → `softmax([0, ..., 0])` is the uniform distribution → every prediction is `1 / num_classes` → no learning signal differentiated across classes.

**Diagnosing it.** Symptoms of a stray final-layer ReLU:
- Training accuracy plateaus at `1 / num_classes`.
- The model's logits are non-negative *everywhere* (a histogram never shows negatives).
- Loss never drops below `log(num_classes)`.

**The fix.** Strip the final ReLU. The pattern is `Linear → ReLU → ... → Linear → ReLU → Linear` (no activation on the very last linear). For *binary* classification with `BCEWithLogitsLoss` the same rule applies: feed raw logits in, never ReLU'd.

### Exercise 1 — diagnose and strip a stray ReLU on the classifier head

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Evaluate
> LO: Evaluate a broken CNN-classifier whose final layer has a stray ReLU; diagnose the failure mode (no negative logits → biased softmax) and return a fixed module that drops the final ReLU.
> Keywords: relu, classifier, logits, debugging
> ```

**KCs targeted:** `final-layer-no-activation`, `logits-vs-probs-pipeline`

You are given a small classifier `BrokenClassifier` (instantiated in the test cell) whose architecture is:
```
Linear(in_features=8, out_features=16) → ReLU →
Linear(in_features=16, out_features=4) → ReLU  ← STRAY!
```
The final ReLU clips all negative logits to 0. With random init, many logits end up exactly 0; downstream `F.cross_entropy` then computes near-uniform probabilities → loss never drops below `log(4) ≈ 1.386`.

Implement `ex1_fix_classifier(broken)`. Given the broken module, **return a new `nn.Module`** that:

1. **Reuses the broken model's weight tensors** (you may copy `broken.fc1.weight.data` / `broken.fc1.bias.data` etc., or reuse the modules directly — your call). Do NOT re-initialize.
2. **Drops the final ReLU.** The forward pass must be `fc1 → ReLU → fc2` — no activation after `fc2`.
3. Has the same input/output shape contract as `broken`.

**Hint.** The simplest fix is a custom `nn.Module` that holds references to `broken.fc1` and `broken.fc2` and applies `F.relu` only between them. Then return an instance of that class.

The test confirms:
- The fixed model produces **at least some negative outputs** for random input (proving the final ReLU is gone).
- Cross-entropy loss on a random-label batch is strictly LOWER than the broken model's loss after a few SGD steps (proving the model can actually learn now).

In [ ]:
def ex1_fix_classifier(broken):
    from torch import nn
    from torch.nn import functional as F

    class FixedClassifier(nn.Module):
        def __init__(self, fc1, fc2):
            super().__init__()
            self.fc1 = fc1   # REUSE — no re-init
            self.fc2 = fc2

        def forward(self, x):
            h = F.relu(self.fc1(x))
            return self.fc2(h)        # NO final ReLU — return raw logits

    return FixedClassifier(broken.fc1, broken.fc2)


<details><summary>Solution</summary>

```python
def ex1_fix_classifier(broken):
    from torch import nn
    from torch.nn import functional as F

    class FixedClassifier(nn.Module):
        def __init__(self, fc1, fc2):
            super().__init__()
            self.fc1 = fc1   # REUSE — no re-init
            self.fc2 = fc2

        def forward(self, x):
            h = F.relu(self.fc1(x))
            return self.fc2(h)        # NO final ReLU — return raw logits

    return FixedClassifier(broken.fc1, broken.fc2)
```

**Why the broken model plateaus at `log(K)`.** Cross-entropy loss for a uniform prediction over `K` classes is exactly `log(K)` (Shannon entropy of the uniform distribution). When every logit is non-negative and many are zero, softmax produces a near-uniform distribution → loss `≈ log(4) ≈ 1.386` for `K=4`. The model literally cannot learn to discriminate, because gradients through the final ReLU are zero for half its inputs.

**Why reuse the modules.** Re-initializing `fc1`/`fc2` would give the fixed model a head start over `broken_train` purely from the new init — masking the real cause. Reusing means both models start from identical weights; any difference at training-end comes from the architectural change only.

**The architectural rule.** For classification, the canonical pattern is:
```
[Linear → activation] x N   →   Linear   →   loss(logits, labels)
```
The final `Linear` outputs **logits** (any real number, possibly negative). `F.cross_entropy` does the `LogSoftmax + NLLLoss` fused internally. Applying ReLU before `cross_entropy` is the most common 'why isn't my model learning' bug after wrong learning rate.

**Symmetric case for regression.** A regression network outputting only non-negative values (with ReLU on the final layer) can never produce negative predictions — useful for things like predicting counts, but a silent bug if your target actually has negatives.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()